In [1]:
import os
import sys
from dotenv import load_dotenv
load_dotenv()

target_folder = "src"
current_dir = os.path.dirname(os.path.abspath("__file__"))

while os.path.basename(current_dir) != target_folder:
    parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
    if parent_dir == current_dir:
        raise FileNotFoundError(f"{target_folder} not found in the directory tree.")
    current_dir = parent_dir

os.chdir(current_dir)
sys.path.insert(0, current_dir)
print(f"Working directory: {os.getcwd()}")

Working directory: /mnt/ffs24/home/chidurah/NCEAS_Unsupervised_NLP/src


In [2]:
import re
import json
import ast
import pickle
import hashlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
from sentence_transformers import SentenceTransformer
from scipy.cluster.hierarchy import fcluster, to_tree, linkage
from sklearn.metrics import adjusted_rand_score, rand_score, adjusted_mutual_info_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

import phate
import pacmap
from cuml.decomposition import PCA as cuPCA
from cuml.manifold import TSNE as cuTSNE
from cuml.manifold import UMAP as cuUMAP
from cuml.cluster import HDBSCAN as cuHDBSCAN

from custom_packages.fowlkes_mallows import FowlkesMallows
from custom_packages.dendrogram_purity import dendrogram_purity
from custom_packages.lca_f1 import lca_f1, clusternode_to_anytree
from run_models.benchmark_datasets.build_ground_truth_trees import build_ground_truth_tree

warnings.filterwarnings("ignore")
np.random.seed(67)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
# ── Parameter grids ──────────────────────────────────────────────────────────

UMAP_GRID = [
    {"n_components": nc, "n_neighbors": nn, "min_dist": md}
    for nc in [50, 100, 300]
    for nn in [5, 15, 30]
    for md in [0.01, 0.1, 0.5]
]

PHATE_GRID = [
    {"n_components": nc, "decay": d}
    for nc in [50, 100, 300]
    for d in [10, 20, 40]
]

PCA_GRID   = [{"n_components": nc} for nc in [50, 100, 300]]
PaCMAP_GRID = [{"n_components": nc} for nc in [50, 100, 300]]
TSNE_GRID  = [{"n_components": 2}]

REDUCTION_GRIDS = {
    "UMAP":   UMAP_GRID,
    "PHATE":  PHATE_GRID,
    "PCA":    PCA_GRID,
    "PaCMAP": PaCMAP_GRID,
    "tSNE":   TSNE_GRID,
}

CLUSTER_METHODS  = ["Agglomerative", "HDBSCAN"]
EMBEDDING_MODELS = [
    "Qwen/Qwen3-Embedding-0.6B",
    "sentence-transformers/all-MiniLM-L6-v2",
]

SWEEP_RESULTS_PATH = "results/rcv1_param_sweep_scores.csv"
FORCE_RERUN = False   # set True to ignore cached results and re-run everything

total_combos = sum(len(g) for g in REDUCTION_GRIDS.values()) * len(EMBEDDING_MODELS) * len(CLUSTER_METHODS)
print(f"Total (reduction × embedding × cluster) combos: {total_combos}")

Total (reduction × embedding × cluster) combos: 172


In [4]:
# ── Helper functions ─────────────────────────────────────────────────────────

def params_to_key(params):
    return hashlib.md5(json.dumps(params, sort_keys=True).encode()).hexdigest()[:8]


def run_reduction(method, params, embeddings, cache_path):
    if os.path.exists(cache_path):
        return np.load(cache_path)
    if method == "UMAP":
        result = cuUMAP(**params).fit_transform(embeddings)
    elif method == "PHATE":
        result = phate.PHATE(n_jobs=-2, random_state=67, t="auto", n_pca=None, **params).fit_transform(embeddings)
    elif method == "PCA":
        result = cuPCA(**params).fit_transform(embeddings)
    elif method == "PaCMAP":
        result = pacmap.PaCMAP(random_state=67, **params).fit_transform(embeddings)
    elif method == "tSNE":
        result = cuTSNE(**params).fit_transform(embeddings)
    if hasattr(result, "to_output"):
        result = result.to_output("numpy")
    elif not isinstance(result, np.ndarray):
        result = np.array(result)
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    np.save(cache_path, result)
    return result


def build_linkage(cluster_method, embed_data, cache_path):
    if os.path.exists(cache_path):
        Z = np.load(cache_path)
    else:
        if cluster_method == "Agglomerative":
            Z = linkage(embed_data, method="ward")
        elif cluster_method == "HDBSCAN":
            model = cuHDBSCAN(min_cluster_size=5, min_samples=1)
            model.fit(embed_data)
            Z = model.single_linkage_tree_.to_numpy()
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        np.save(cache_path, Z)
    tree, _ = to_tree(Z, rd=True)
    return Z, tree


def evaluate_combo(embed_data, cluster_method, cluster_levels, topic_dict, gt_tree_root, linkage_cache_path):
    Z, tree = build_linkage(cluster_method, embed_data, linkage_cache_path)
    pred_tree = clusternode_to_anytree(tree)
    rows = []
    for level in cluster_levels:
        labels = fcluster(Z, level, criterion="maxclust")
        closest_level = min(topic_dict.keys(), key=lambda k: abs(k - level))
        topic_series = topic_dict[closest_level]
        valid_idx = ~pd.isna(topic_series)
        target_lst = topic_series[valid_idx]
        label_lst  = labels[valid_idx]
        try:
            fm = FowlkesMallows.Bk({level: target_lst}, {level: label_lst})[level]["FM"]
        except Exception:
            fm = np.nan
        rows.append({
            "level":             level,
            "FM":                fm,
            "Rand":              rand_score(target_lst, label_lst),
            "ARI":               adjusted_rand_score(target_lst, label_lst),
            "AMI":               adjusted_mutual_info_score(target_lst, label_lst),
            "Dendrogram_Purity": dendrogram_purity(pred_tree, topic_series),
            "LCA_F1":            lca_f1(pred_tree, gt_tree_root, topic_series) if gt_tree_root is not None else np.nan,
        })
    return rows

In [5]:
# ── Load RCV1 + ground truth tree ────────────────────────────────────────────

rcv1 = pd.read_csv("data/rcv1/rcv1.csv")
rcv1 = rcv1.drop_duplicates(subset="topic", keep=False).reset_index(drop=True)
rcv1 = rcv1.drop_duplicates(subset="item_id", keep=False).reset_index(drop=True)
rcv1 = rcv1.dropna().reset_index(drop=True)
rcv1 = rcv1[rcv1["topic"].apply(lambda x: isinstance(x, str) and x.strip() != "")].reset_index(drop=True)
print(f"RCV1 shape: {rcv1.shape}")

gt_df = rcv1.rename(columns={c: c.replace(" ", "_") for c in rcv1.columns})
gt_tree_root, _ = build_ground_truth_tree(gt_df, depth=2)

topic_dict = {}
for col in rcv1.columns:
    if re.match(r"^category[_ ]\d+$", col):
        topic_dict[len(rcv1[col].unique())] = np.array(rcv1[col])

cluster_levels = sorted(topic_dict.keys())
print(f"Cluster levels: {cluster_levels}")

RCV1 shape: (1566, 4)
Cluster levels: [58, 72]


In [6]:
# ── Generate embeddings ───────────────────────────────────────────────────────
# Change embedding_model here to switch between the two

embedding_model = "Qwen/Qwen3-Embedding-0.6B"
# embedding_model = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(
    embedding_model,
    model_kwargs={"attn_implementation": "sdpa", "device_map": "auto"} if "Qwen" in embedding_model else {},
    tokenizer_kwargs={"padding_side": "left"} if "Qwen" in embedding_model else {},
)
embeddings = model.encode(
    rcv1["topic"].tolist(),
    batch_size=8,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"Embeddings shape: {embeddings.shape}")

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

Embeddings shape: (1566, 1024)


In [7]:
# ── Run parameter sweep ───────────────────────────────────────────────────────

all_rows = []

combos = [
    (method, params)
    for method, grid in REDUCTION_GRIDS.items()
    for params in grid
]

for reduction_method, reduction_params in tqdm(combos, desc="Reductions"):
    try:
        if reduction_method == "UMAP":
            reduced = cuUMAP(**reduction_params).fit_transform(embeddings)
        elif reduction_method == "PHATE":
            reduced = phate.PHATE(n_jobs=-2, random_state=67, t="auto", n_pca=None, **reduction_params).fit_transform(embeddings)
        elif reduction_method == "PCA":
            reduced = cuPCA(**reduction_params).fit_transform(embeddings)
        elif reduction_method == "PaCMAP":
            reduced = pacmap.PaCMAP(random_state=67, **reduction_params).fit_transform(embeddings)
        elif reduction_method == "tSNE":
            reduced = cuTSNE(**reduction_params).fit_transform(embeddings)

        if hasattr(reduced, "to_output"):
            reduced = reduced.to_output("numpy")
        elif not isinstance(reduced, np.ndarray):
            reduced = np.array(reduced)

    except Exception as e:
        print(f"ERROR in {reduction_method} {reduction_params}: {e}")
        continue

    for cluster_method in CLUSTER_METHODS:
        try:
            if cluster_method == "Agglomerative":
                Z = linkage(reduced, method="ward")
            elif cluster_method == "HDBSCAN":
                hdb = cuHDBSCAN(min_cluster_size=5, min_samples=1)
                hdb.fit(reduced)
                Z = hdb.single_linkage_tree_.to_numpy()

            tree, _ = to_tree(Z, rd=True)
            pred_tree = clusternode_to_anytree(tree)

            for level in cluster_levels:
                labels = fcluster(Z, level, criterion="maxclust")
                closest_level = min(topic_dict.keys(), key=lambda k: abs(k - level))
                topic_series = topic_dict[closest_level]
                valid_idx = ~pd.isna(topic_series)
                target_lst = topic_series[valid_idx]
                label_lst  = labels[valid_idx]

                try:
                    fm = FowlkesMallows.Bk({level: target_lst}, {level: label_lst})[level]["FM"]
                except Exception:
                    fm = np.nan

                all_rows.append({
                    "embedding_model":   embedding_model,
                    "reduction_method":  reduction_method,
                    "reduction_params":  str(reduction_params),
                    "cluster_method":    cluster_method,
                    "cluster_params":    "{}",
                    "level":             level,
                    "FM":                fm,
                    "Rand":              rand_score(target_lst, label_lst),
                    "ARI":               adjusted_rand_score(target_lst, label_lst),
                    "AMI":               adjusted_mutual_info_score(target_lst, label_lst),
                    "Dendrogram_Purity": dendrogram_purity(pred_tree, topic_series),
                    "LCA_F1":            lca_f1(pred_tree, gt_tree_root, topic_series),
                })

        except Exception as e:
            print(f"  ERROR in {cluster_method}: {e}")
            continue

bertopic_result_df = pd.DataFrame(all_rows)
os.makedirs("results", exist_ok=True)
bertopic_result_df.to_csv(SWEEP_RESULTS_PATH, index=False)
print(f"Sweep complete. {len(bertopic_result_df)} rows saved to {SWEEP_RESULTS_PATH}")

Reductions:  63%|██████▎   | 27/43 [03:26<02:06,  7.91s/it]

Calculating PHATE...
  Running PHATE on 1566 observations and 1024 variables.
  Calculating graph and diffusion operator...
    Calculating KNN search...
    Calculated KNN search in 0.77 seconds.
    Calculating affinities...
    Calculated affinities in 0.89 seconds.
  Calculated graph and diffusion operator in 1.69 seconds.
  Calculating optimal t...
    Automatically selected t = 16
  Calculated optimal t in 0.49 seconds.
  Calculating diffusion potential...
  Calculated diffusion potential in 0.10 seconds.
  Calculating metric MDS...
    SGD-MDS may not have converged: stress changed by 7.6% in final iterations. Consider increasing n_iter or adjusting learning_rate.
  Calculated metric MDS in 21.02 seconds.
Calculated PHATE in 23.36 seconds.


Reductions:  65%|██████▌   | 28/43 [03:56<03:40, 14.73s/it]

Calculating PHATE...
  Running PHATE on 1566 observations and 1024 variables.
  Calculating graph and diffusion operator...
    Calculating KNN search...
    Calculated KNN search in 0.75 seconds.
    Calculating affinities...
    Calculated affinities in 0.28 seconds.
  Calculated graph and diffusion operator in 1.04 seconds.
  Calculating optimal t...
    Automatically selected t = 34
  Calculated optimal t in 0.49 seconds.
  Calculating diffusion potential...
  Calculated diffusion potential in 0.14 seconds.
  Calculating metric MDS...
    SGD-MDS may not have converged: stress changed by -2.3% in final iterations. Consider increasing n_iter or adjusting learning_rate.
  Calculated metric MDS in 20.91 seconds.
Calculated PHATE in 22.61 seconds.


Reductions:  67%|██████▋   | 29/43 [04:27<04:32, 19.43s/it]

...

Calculating PHATE...
  Running PHATE on 1566 observations and 1024 variables.
  Calculating graph and diffusion operator...
    Calculating KNN search...
    Calculated KNN search in 0.73 seconds.
    Calculating affinities...
    Calculated affinities in 0.26 seconds.
  Calculated graph and diffusion operator in 1.00 seconds.
  Calculating optimal t...
    Automatically selected t = 34
  Calculated optimal t in 0.47 seconds.
  Calculating diffusion potential...
  Calculated diffusion potential in 0.14 seconds.
  Calculating metric MDS...
    SGD-MDS may not have converged: stress changed by -2.2% in final iterations. Consider increasing n_iter or adjusting learning_rate.
  Calculated metric MDS in 151.44 seconds.
Calculated PHATE in 153.09 seconds.


Reductions:  81%|████████▏ | 35/43 [12:58<13:43, 102.93s/it]

Calculating PHATE...
  Running PHATE on 1566 observations and 1024 variables.
  Calculating graph and diffusion operator...
    Calculating KNN search...
    Calculated KNN search in 0.74 seconds.
    Calculating affinities...
    Calculated affinities in 0.05 seconds.
  Calculated graph and diffusion operator in 0.80 seconds.
  Calculating optimal t...
    Automatically selected t = 19
  Calculated optimal t in 0.48 seconds.
  Calculating diffusion potential...
  Calculated diffusion potential in 0.14 seconds.
  Calculating metric MDS...
    SGD-MDS may not have converged: stress changed by -1.5% in final iterations. Consider increasing n_iter or adjusting learning_rate.
  Calculated metric MDS in 149.43 seconds.
Calculated PHATE in 150.87 seconds.


Reductions:  91%|█████████ | 39/43 [16:00<03:04, 46.16s/it]Warning: random state is set to 67.
Note: `n_components != 2` have not been thoroughly tested.
Reductions:  93%|█████████▎| 40/43 [16:11<01:47, 35.71s/it]Warning: random state is set to 67.
Note: `n_components != 2` have not been thoroughly tested.
Reductions:  95%|█████████▌| 41/43 [16:25<00:58, 29.06s/it]Warning: random state is set to 67.
Note: `n_components != 2` have not been thoroughly tested.
Reductions: 100%|██████████| 43/43 [16:45<00:00, 23.38s/it]

Sweep complete. 172 rows saved to results/rcv1_param_sweep_scores.csv


In [ ]:
# Impute default parameters for analysis
replacement_map = {
    "{}": "{'n_components': 3, 'min_dist': 0.1, 'n_neighbors': 15}",
    "{'n_components': 300}": "{'n_components': 300, 'min_dist': 0.1, 'n_neighbors': 15}",
    "{'n_components': 100}": "{'n_components': 100, 'min_dist': 0.1, 'n_neighbors': 15}"
}

# Apply changes in-place to rows where reduction_method is 'UMAP'
bertopic_result_df.loc[
    bertopic_result_df['reduction_method'] == 'UMAP', 'reduction_params'
] = bertopic_result_df.loc[
    bertopic_result_df['reduction_method'] == 'UMAP', 'reduction_params'
].apply(lambda x: replacement_map.get(str(x), x))


In [ ]:
def analyze_feature_statistics(df, target_col, predictor_cols):
    """
    Analyzes feature relevance using statistical associations with the target column.

    Parameters:
    - df: Pandas DataFrame containing the data
    - target_col: String, name of the target column
    - predictor_cols: List of strings, names of predictor columns

    Returns:
    - feature_scores: DataFrame with statistical association scores
    - optimal_values: Dictionary mapping features to optimal values (based on max target mean)
    """
    df = df.dropna(subset=[target_col])
    categorical_cols = [col for col in predictor_cols if df[col].dtype == 'object']
    numeric_cols = [col for col in predictor_cols if col not in categorical_cols]

    feature_scores = []
    optimal_values = {}

    # Handle numeric features
    for col in numeric_cols:
        temp_df = df[[col, target_col]].dropna()
        if temp_df.empty:
            continue
        corr = temp_df[col].corr(temp_df[target_col])
        grouped = temp_df.groupby(col)[target_col].mean()
        if not grouped.empty:
            opt_val = grouped.idxmax()
        else:
            opt_val = None
        feature_scores.append({'Feature': col, 'Score': abs(corr), 'Type': 'Numeric'})
        optimal_values[col] = opt_val

    # Handle categorical features
    for col in categorical_cols:
        df[col] = df[col].astype(str).fillna("Missing")
        grouped = df.groupby(col)[target_col].mean()
        if not grouped.empty:
            opt_val = grouped.idxmax()
            score = grouped.max() - grouped.min()
        else:
            opt_val = None
            score = 0
        feature_scores.append({'Feature': col, 'Score': score, 'Type': 'Categorical'})
        optimal_values[col] = opt_val

    feature_scores_df = pd.DataFrame(feature_scores).sort_values(by='Score', ascending=False)

    return feature_scores_df, optimal_values

In [ ]:
def analyze_feature_importance(df, target_col, predictor_cols, n_estimators=100, random_state=42, plot_violin=False):
    """
    Trains a Random Forest model to evaluate feature importance and optimal value ranges.
    
    Parameters:
    - df: Pandas DataFrame containing the data
    - target_col: String, name of the target column
    - predictor_cols: List of strings, names of predictor columns
    - n_estimators: Number of trees in the Random Forest (default: 100)
    - random_state: Random seed for reproducibility (default: 42)
    - plot_violin: Boolean, whether to plot violin plots for each predictor (default: False)
    
    Returns:
    - feature_importance: DataFrame with feature importance scores
    - optimal_ranges: Dictionary mapping features to optimal value estimates
    """
    # Keep NaNs in predictors but drop NaNs in the target column
    df = df.dropna(subset=[target_col])
    
    # Separate categorical and numeric columns
    categorical_cols = [col for col in predictor_cols if df[col].dtype == 'object']
    numeric_cols = [col for col in predictor_cols if col not in categorical_cols]
    # Copy dataframe for encoding
    df_encoded = df.copy()
    label_encoders = {}

    # Encode categorical variables (treat NaNs as "Missing")
    for col in categorical_cols:
        df_encoded[col] = df_encoded[col].astype(str).fillna("Missing")  # Convert NaN to string
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col])
        label_encoders[col] = le


    # Handle NaNs in numeric columns by replacing them with a placeholder (-99999)
    df_encoded[numeric_cols] = df_encoded[numeric_cols].dropna()

    # Define features and target
    X = df_encoded[predictor_cols]
    y = df[target_col]
    
    # Train Random Forest model
    rf = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state)
    rf.fit(X, y)
    
    # Compute feature importance
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': rf.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    
    # Plot feature importance
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Importance', y='Feature', data=feature_importance)
    plt.title('Feature Importance in Random Forest')
    plt.show()
    
    # Compute optimal ranges for features
    optimal_ranges = {}
    optimal_scores = {}
    for feature in predictor_cols:
        feature_vals = np.sort(df_encoded[feature].dropna().unique())  # Sorted unique values from encoded df
        if len(feature_vals) == 0:
            optimal_ranges[feature] = None
            continue
        
        pred_vals = [rf.predict(X[df_encoded[feature] == val]) for val in feature_vals]
        max_pred_idx = np.argmax([np.mean(pred) for pred in pred_vals])
        optimal_value = feature_vals[max_pred_idx]
        optimal_score = np.max(pred_vals[max_pred_idx])
        optimal_scores[feature] = optimal_score
        

        # Decode categorical variables back to original values
        if feature in label_encoders:
            optimal_value = label_encoders[feature].inverse_transform([optimal_value])[0]
        
        optimal_ranges[feature] = optimal_value
    # Optionally plot violin plots
    if plot_violin:
        for feature in predictor_cols:
            plt.figure(figsize=(10, 5))
            if feature in categorical_cols:
                df_plot = df.copy()
                df_plot[feature] = df_plot[feature].astype(str).fillna("Missing")  # Keep NaNs as "Missing"
                sns.violinplot(x=df_plot[feature], y=target_col, data=df_plot)
            else:
                df_plot = df.copy()
                df_plot[feature] = df_plot[feature].fillna('None')  # Keep NaNs explicitly
                sns.violinplot(x=df_plot[feature].astype(str), y=target_col, data=df_plot)
            
            plt.title(f'Distribution of {target_col} by {feature}')
            plt.xlabel(feature)
            plt.ylabel(target_col)
            plt.xticks(rotation=45)
            plt.show()
    
    return feature_importance, optimal_ranges,optimal_scores

def filter_and_expand(df, category, filter_val):
    """
    Filters the DataFrame based on a specific value in the given method column
    and expands the dictionary in the corresponding params column into separate columns.
    
    :param df: DataFrame to process
    :param category: Either 'reduction' or 'cluster' to specify which method to filter
    :param filter_val: Value to filter for
    :return: Processed DataFrame
    """
    method_col = f"{category}_method"
    params_col = f"{category}_params"
    
    # Filter the DataFrame
    filtered_df = df[df[method_col] == filter_val].copy()
    
    # Convert string dictionaries to actual dictionaries (if necessary)
    if filtered_df[params_col].dtype == 'object':
        filtered_df[params_col] = filtered_df[params_col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    # print(filtered_df[params_col])
    # Expand the parameters dictionary into separate columns
    params_df = filtered_df[params_col].apply(pd.Series)
    # params_df = params_df.dropna()
    # Concatenate the expanded parameters with the original DataFrame
    filtered_df = pd.concat([filtered_df.drop(columns=[params_col]), params_df], axis=1)
    
    return filtered_df, params_df.columns

## Compare Methods

In [ ]:
score = 'ARI'
bertopic_result_df[f'mean_{score}'] = bertopic_result_df.groupby(['reduction_method', 'cluster_method', 'reduction_params', 'cluster_params'])[f'{score}'].transform('mean')
feature_importance, optimal_ranges,optimal_scores = analyze_feature_importance(bertopic_result_df, f'mean_{score}',['reduction_method',	'cluster_method'], plot_violin=True)

print(optimal_ranges)
optimal_scores

## Compare parameters for any given Mmthod

In [ ]:
df_param,cols = filter_and_expand(bertopic_result_df, 'reduction', 'UMAP')
df_param1 = df_param
feature_importance, optimal_ranges, optimal_scores = analyze_feature_importance(df_param, f'mean_{score}',cols, plot_violin=True)
print(optimal_scores)
print(optimal_ranges)

In [ ]:
df_param,cols = filter_and_expand(bertopic_result_df, 'reduction', 'PHATE')
df_param1 = df_param
feature_importance, optimal_ranges, optimal_scores = analyze_feature_importance(df_param, f'mean_{score}',cols, plot_violin=True)
print(optimal_scores)
print(optimal_ranges)

In [ ]:
analyze_feature_statistics(df_param,f'mean_{score}',cols)